# Increasing signal to noise ratio via integration gain 



In [79]:
import numpy as np 
import plotly.graph_objects as go
import ipywidgets as widgets

In [80]:
# Sample rate 
Fs = 1.0e4
# Bit Rate 
Fb = 10
# total time 
T = 1.0
seed = 42



In [81]:
def makeSignal( CN0 ):
    global t, signal
    
    rng = np.random.default_rng(seed=seed)                  # seed is optional, for reproducibility
    L = int( Fs * T )

    # Translate CN0 to variance and std

    var = 1.0 # unity variance
    std = np.sqrt( var )
    noise = rng.normal( loc=0.0, scale=std, size=L)
    # bit signal
    samplesPerBit = int( Fs / Fb )
    nBits = L / samplesPerBit
    bits = np.arange( nBits ) % 2
    symbols = 2*bits-1
    A = np.sqrt( 2 * 10**( CN0 / 10 ) / Fs ) # unity variance -> N0 = 2 / Fs, C = A^2 = CN0 * N0 
    signal = A * np.repeat(symbols, samplesPerBit)

    # Time
    t = np.arange( L ) / Fs



    signal = noise + signal

    # Plot the signal
    # t vs signal
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.update_layout( xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

# carrier to noise density - dB/Hz 
CN0 = widgets.FloatSlider( value=30, min=10, max=100, step=1, description="CN0 (dB-Hz)", continuous_update=False )

# one plot, redrawn in place when the slider moves
display( CN0 )
# pass plotly's mimebundle directly - display(fig) calls fig.show() and ignores display_id
plot = display( makeSignal( CN0.value )._repr_mimebundle_(), raw=True, display_id=True )
CN0.observe( lambda change: plot.update( makeSignal( change.new )._repr_mimebundle_(), raw=True ), names="value" )

FloatSlider(value=30.0, continuous_update=False, description='CN0 (dB-Hz)', min=10.0, step=1.0)

## Coherent Integration

In [82]:
# N is a slider for number of samples to integrate over 
def plotCoherent( t , signal, N ):
    B = len(signal) // N                # number of whole blocks of N samples
    signalClipped = signal[:B*N]
    signalCoherent = np.reshape( signalClipped, (B, N) )
    coherent = np.sum( signalCoherent, axis=1 ) /  N
    tClipped = t[:B*N]
    tCoherent = tClipped[::N] # every Nth sample 

    # plot signal and coherent integration signal so we can see
    # - down sampling 
    # - gain
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.add_trace( go.Scattergl( x=tCoherent, y=coherent, mode="markers", name="coherent" ) )
    fig.update_layout( xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

# coherent integration length - samples
# only divisors of samplesPerBit, so no block of N samples straddles a bit edge
samplesPerBit = int( Fs / Fb )
validN = [ n for n in range( 1, samplesPerBit + 1 ) if samplesPerBit % n == 0 ]
N = widgets.SelectionSlider( options=validN, value=validN[2], description="N (samples)", continuous_update=False )

# one plot, redrawn in place when the slider moves
display( N )
coherentPlot = display( plotCoherent( t, signal, N.value )._repr_mimebundle_(), raw=True, display_id=True )
N.observe( lambda change: coherentPlot.update( plotCoherent( t, signal, change.new )._repr_mimebundle_(), raw=True ), names="value" )

SelectionSlider(continuous_update=False, description='N (samples)', index=2, options=(1, 2, 4, 5, 8, 10, 20, 2…

## Non Coherent Integration 

In [ ]:
# M is a slider for number of coherent intervals to sum over 
def plotNonCoherent( t, signal, N, M ):
    # coherent integration over N samples
    B = len(signal) // N                # number of whole blocks of N samples
    signalClipped = signal[:B*N]
    coherent = np.sum( np.reshape( signalClipped, (B, N) ), axis=1 ) / N
    tCoherent = t[:B*N][::N] # every Nth sample 

    # non-coherent integration: square the coherent outputs, then sum M of them
    K = B // M                          # number of whole groups of M coherent outputs
    power = coherent[:K*M]**2
    nonCoherent = np.sum( np.reshape( power, (K, M) ), axis=1 ) / M
    tNonCoherent = tCoherent[:K*M][::M] # every Mth coherent sample

    # plot signal, coherent and non-coherent signal so we can see
    # - squaring removes the bit sign
    # - summing M squared outputs smooths out the noise
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.add_trace( go.Scattergl( x=tCoherent, y=coherent, mode="markers", name="coherent" ) )
    fig.add_trace( go.Scattergl( x=tNonCoherent, y=nonCoherent, mode="markers", name="non-coherent" ) )
    fig.update_layout( xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

# non-coherent integration length - number of coherent intervals
M = widgets.IntSlider( value=1, min=1, max=5, step=1, description="M (intervals)", continuous_update=False )

# one plot, redrawn in place when either slider moves
display( M )
nonCoherentPlot = display( plotNonCoherent( t, signal, N.value, M.value )._repr_mimebundle_(), raw=True, display_id=True )
redrawNonCoherent = lambda change: nonCoherentPlot.update( plotNonCoherent( t, signal, N.value, M.value )._repr_mimebundle_(), raw=True )
M.observe( redrawNonCoherent, names="value" )
N.observe( redrawNonCoherent, names="value" )

IntSlider(value=1, continuous_update=False, description='M (intervals)', max=5, min=1)

## Probability of detection and false alarm